# 🔍 Scout Agent — Draft

Scout is the first agent in the **Seesaw** pipeline. It takes a research question about AI safety / mechanistic interpretability and produces a structured **Research Plan** that the Lens agent will execute.

## Pipeline position

```
User
 │
 ▼
[Research Question]
 │
 ▼
┌─────────────────────────────────────┐
│              Scout                  │  ◄── YOU ARE HERE
│                                     │
│  search_arxiv()                     │
│  search_web()      ─── Firecrawl    │
│  scrape_url()      ─── Firecrawl    │
│  save_research_plan()               │
└─────────────────────────────────────┘
 │
 ▼
[Research Plan]  →  Lens  →  Quill  →  Report
```

## Stack
- **LangGraph** — agent loop and state management (`create_react_agent`)
- **Claude** — backbone LLM
- **arxiv** — academic paper search (no API key needed)
- **Firecrawl** — web search + deep URL scraping (mirrors Nova's pattern)

## 1. Setup

In [ ]:
%pip install -q \
    langchain-anthropic \
    langgraph \
    arxiv \
    firecrawl-py \
    python-dotenv

In [ ]:
import os
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

import arxiv
from firecrawl import FirecrawlApp

load_dotenv()

print("✅ Imports OK")

In [ ]:
# ── API Keys ──────────────────────────────────────────────────────────────────
ANTHROPIC_API_KEY  = os.getenv("ANTHROPIC_API_KEY")
FIRECRAWL_API_KEY  = os.getenv("FIRECRAWL_API_KEY")

# ── Paths ─────────────────────────────────────────────────────────────────────
SCOUT_ROOT  = Path(".")
OUTPUTS_DIR = SCOUT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

# ── Status ────────────────────────────────────────────────────────────────────
print(f"Anthropic API key  : {'✅ set' if ANTHROPIC_API_KEY else '❌ missing — required'}")
print(f"Firecrawl API key  : {'✅ set' if FIRECRAWL_API_KEY else '❌ missing — required for search_web and scrape_url'}")
print(f"Outputs directory  : {OUTPUTS_DIR.resolve()}")

## 2. Scout Tools

Scout has five tools. Firecrawl handles all web I/O — consistent with Nova's pattern.

| Tool | Purpose | Backed by |
|---|---|---|
| `search_arxiv` | Search academic papers via the official arXiv API — structured, reliable | arxiv library (free) |
| `search_arxiv_web` | Search arxiv.org via Firecrawl — web-ranked, surfaces recent preprints | Firecrawl search |
| `search_web` | Search the broader web for blog posts, GitHub repos, Colab notebooks | Firecrawl search |
| `scrape_url` | Get full cleaned markdown from any URL | Firecrawl scrape |
| `save_research_plan` | Write the final Research Plan to disk | local file I/O |

**When to use which arxiv tool:**
- `search_arxiv` → structured keyword queries, known paper titles, getting clean abstracts
- `search_arxiv_web` → web-ranked results, recent preprints the API buries, finding related survey posts

In [ ]:
@tool
def search_arxiv(query: str, max_results: int = 8) -> str:
    """Search arXiv for academic papers on mechanistic interpretability and AI safety.
    Use multiple targeted queries to cover different aspects of the research question.
    Best for finding foundational papers, circuit analysis, and formal methods.

    Args:
        query: The search query (e.g. 'GPT-2 indirect object identification circuit')
        max_results: Number of papers to return (default: 8)

    Returns:
        Formatted list of papers with titles, authors, abstracts, and URLs
    """
    client = arxiv.Client()
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance,
    )

    results = []
    for paper in client.results(search):
        authors = ", ".join(a.name for a in paper.authors[:3])
        if len(paper.authors) > 3:
            authors += " et al."
        results.append(
            f"**Title**: {paper.title}\n"
            f"**Authors**: {authors}\n"
            f"**Published**: {paper.published.strftime('%Y-%m-%d')}\n"
            f"**Abstract**: {paper.summary[:600]}...\n"
            f"**URL**: {paper.entry_id}\n"
        )

    if not results:
        return f"No papers found for query: '{query}'"

    return f"Found {len(results)} papers for '{query}':\n\n" + "\n---\n".join(results)


print("✅ search_arxiv defined")

In [ ]:
@tool
def search_arxiv_web(query: str, limit: int = 5) -> str:
    """Search arxiv.org via Firecrawl for papers on mechanistic interpretability and AI safety.
    Complements search_arxiv: while search_arxiv uses the official API (structured, keyword-matched),
    this tool uses web search ranking — better for surfacing recent preprints, finding papers
    that cite a key work, or when the API's relevance ranking buries what you need.

    Args:
        query: The search query (e.g. 'transformer circuits attention head superposition')
        limit: Number of results to return (default: 5)

    Returns:
        Search results from arxiv.org with titles, URLs, and content snippets
    """
    if not FIRECRAWL_API_KEY:
        return "search_arxiv_web unavailable: FIRECRAWL_API_KEY not set."

    app = FirecrawlApp(api_key=FIRECRAWL_API_KEY)

    # Scope the search to arxiv.org
    arxiv_query = f"site:arxiv.org {query}"

    try:
        response = app.search(arxiv_query, limit=limit)

        raw_results = getattr(response, "data", None) or response
        if isinstance(raw_results, dict):
            raw_results = raw_results.get("data", [])

        if not raw_results:
            return f"No arxiv.org results found for: '{query}'"

        results = []
        for r in raw_results:
            if isinstance(r, dict):
                title   = r.get("title", "N/A")
                url     = r.get("url", "N/A")
                snippet = r.get("description") or r.get("markdown", "")
            else:
                title   = getattr(r, "title", "N/A")
                url     = getattr(r, "url", "N/A")
                snippet = getattr(r, "description", None) or getattr(r, "markdown", "") or ""

            # Only keep actual arxiv.org links
            if "arxiv.org" not in str(url):
                continue

            results.append(
                f"**Title**: {title}\n"
                f"**URL**: {url}\n"
                f"**Snippet**: {str(snippet)[:400]}...\n"
            )

        if not results:
            return f"No arxiv.org results found for: '{query}'"

        return f"arxiv.org web results for '{query}':\n\n" + "\n---\n".join(results)

    except Exception as e:
        return f"search_arxiv_web failed for '{query}': {str(e)}"


print("✅ search_arxiv_web defined")

In [ ]:
@tool
def search_web(query: str, limit: int = 5) -> str:
    """Search the web for recent blog posts, GitHub repos, Colab notebooks,
    and community resources on AI safety and mechanistic interpretability.
    Returns URLs and content snippets. Use scrape_url to get full content
    from the most relevant results.

    Args:
        query: The search query
        limit: Number of results to return (default: 5)

    Returns:
        Search results with titles, URLs, and content snippets
    """
    if not FIRECRAWL_API_KEY:
        return "search_web unavailable: FIRECRAWL_API_KEY not set."

    app = FirecrawlApp(api_key=FIRECRAWL_API_KEY)

    try:
        response = app.search(query, limit=limit)

        # response.data is a list of SearchResult objects
        raw_results = getattr(response, "data", None) or response
        if isinstance(raw_results, dict):
            raw_results = raw_results.get("data", [])

        if not raw_results:
            return f"No web results found for: '{query}'"

        results = []
        for r in raw_results:
            # Handle both object and dict responses across SDK versions
            if isinstance(r, dict):
                title   = r.get("title", "N/A")
                url     = r.get("url", "N/A")
                snippet = r.get("description") or r.get("markdown", "")[:400]
            else:
                title   = getattr(r, "title", "N/A")
                url     = getattr(r, "url", "N/A")
                snippet = (getattr(r, "description", None) or getattr(r, "markdown", "") or "")[:400]

            results.append(
                f"**Title**: {title}\n"
                f"**URL**: {url}\n"
                f"**Snippet**: {snippet}...\n"
            )

        return f"Web results for '{query}':\n\n" + "\n---\n".join(results)

    except Exception as e:
        return f"search_web failed for '{query}': {str(e)}"


print("✅ search_web defined")

In [ ]:
# Firecrawl cache setting — mirrors Nova's scraping_handler.py
# maxAge=1 week gives ~500% faster scraping for static content (docs, papers, blogs)
_MAX_AGE_ONE_WEEK = 604_800_000  # milliseconds
_MAX_RETRIES      = 3
_MAX_CHARS        = 8_000        # truncate to keep context window manageable


@tool
def scrape_url(url: str) -> str:
    """Scrape and extract the full cleaned markdown content from a URL using Firecrawl.
    Use this after search_web to get the complete content of a specific page —
    full paper text, entire blog post, complete README, etc.

    Args:
        url: The URL to scrape

    Returns:
        Cleaned markdown content from the page (truncated at 8000 chars)
    """
    if not FIRECRAWL_API_KEY:
        return "scrape_url unavailable: FIRECRAWL_API_KEY not set."

    app = FirecrawlApp(api_key=FIRECRAWL_API_KEY)

    for attempt in range(_MAX_RETRIES):
        try:
            res = app.scrape_url(
                url,
                formats=["markdown"],
                # maxAge: use cached version if available — much faster
                # same strategy as Nova's scraping_handler.py
            )

            # Handle both object and dict responses across SDK versions
            if isinstance(res, dict):
                title    = res.get("metadata", {}).get("title", "N/A")
                markdown = res.get("markdown", "")
            else:
                title    = getattr(getattr(res, "metadata", None), "title", "N/A") or "N/A"
                markdown = getattr(res, "markdown", "") or ""

            if not markdown.strip():
                return f"No content returned for {url}"

            # Truncate to avoid blowing up context window
            if len(markdown) > _MAX_CHARS:
                markdown = markdown[:_MAX_CHARS] + "\n\n[... content truncated ...]"

            return f"# {title}\nSource: {url}\n\n{markdown}"

        except Exception as e:
            if attempt < _MAX_RETRIES - 1:
                import time
                delay = 5 * (2 ** attempt)  # exponential backoff: 5s, 10s
                print(f"  ⚠️ scrape attempt {attempt + 1} failed for {url} — retrying in {delay}s")
                time.sleep(delay)
            else:
                return f"Failed to scrape {url} after {_MAX_RETRIES} attempts: {str(e)}"

    return f"Failed to scrape {url}"


print("✅ scrape_url defined")

In [ ]:
@tool
def save_research_plan(plan: str, filename: str = "research_plan.md") -> str:
    """Save the final structured Research Plan to disk.
    Call this ONLY when you have gathered sufficient information and are ready
    to produce the final output. This signals the end of the Scout workflow.

    Args:
        plan: The complete research plan in markdown format
        filename: Output filename (default: research_plan.md)

    Returns:
        Confirmation message with the saved file path
    """
    output_path = OUTPUTS_DIR / filename
    output_path.write_text(plan, encoding="utf-8")
    return (
        f"✅ Research plan saved to: {output_path.resolve()}\n\n"
        f"Plan preview (first 500 chars):\n{plan[:500]}..."
    )


print("✅ save_research_plan defined")

In [ ]:
# All tools registered in one place — easy to extend later
SCOUT_TOOLS = [
    search_arxiv,
    search_arxiv_web,
    search_web,
    scrape_url,
    save_research_plan,
]

print(f"✅ {len(SCOUT_TOOLS)} tools registered: {[t.name for t in SCOUT_TOOLS]}")

In [ ]:
SCOUT_SYSTEM_PROMPT = """
You are Scout, an AI safety research agent specialised in mechanistic interpretability.

Your job: take a research question and produce a comprehensive Research Plan by gathering
information from academic papers, blog posts, GitHub repositories, and documentation.
The Research Plan will be handed to the Lens agent, which will run the experiments you specify.

## Workflow

Follow these steps in order:

1. **Decompose** — break the research question into 3-5 sub-topics
2. **Search arXiv** — run at least 3 targeted searches using both search_arxiv and search_arxiv_web
3. **Search the web** — find blog posts, GitHub repos, and Colab notebooks via search_web
4. **Scrape key URLs** — use scrape_url to get the full content of the 2-3 most relevant pages
5. **Synthesise** — identify patterns, hypotheses, and open questions
6. **Save** — call `save_research_plan` with the final structured plan

## Research Plan Format

Your final output passed to `save_research_plan` MUST follow this exact structure:

```
# Research Plan: <research_question>

**Date**: <date>
**Status**: Draft

---

## Background
<2-3 paragraphs summarising the current state of knowledge on this topic>

## Key Papers & Resources
<bullet list — title, authors, year, one-line description, URL>

## Hypotheses
<numbered list of specific, testable hypotheses>

## Proposed Experiments
<For each experiment, use this exact format:>

### Experiment N: <name>
- **Lens Tool**: <one of: logit_lens | attention_pattern | ablation | activation_patching | direct_logit_attribution>
- **Model**: <e.g. gpt2-small, pythia-160m>
- **Dataset / Prompts**: <what inputs to use>
- **What to measure**: <specific metric or observation>
- **Hypothesis tested**: <which hypothesis above this tests>
- **Expected outcome**: <what you predict>

## Target Models
<which models and why — focus on TransformerLens-compatible small models>

## Expected Findings
<what the experiments are likely to reveal>

## Open Questions
<what remains unknown after this research plan is executed>
```

## Available Lens Tools (for Proposed Experiments)

| Tool | What it does |
|---|---|
| `logit_lens` | Visualises how predictions evolve across layers |
| `attention_pattern` | Shows which tokens attend to which at each layer |
| `ablation` | Zero/mean ablates heads or MLPs to measure their importance |
| `activation_patching` | Patches activations to find causal components |
| `direct_logit_attribution` | Decomposes final logit output by layer and component |

## Scout Tool Guide

| Tool | When to use |
|---|---|
| `search_arxiv` | Structured keyword queries, known paper titles, clean abstracts |
| `search_arxiv_web` | Web-ranked arXiv results, recent preprints, papers the API buries |
| `search_web` | Blog posts, GitHub repos, Colab notebooks, community resources |
| `scrape_url` | Full content of a specific page after finding it via search |

## Guidelines
- Focus on mechanistic interpretability and AI safety implications
- Prioritise papers from 2022 onwards (the modern mech interp era)
- Be concrete about experiments — Lens will run exactly what you specify
- Each hypothesis must be testable with TransformerLens on small models
- Prefer GPT-2 Small and Pythia models for experiments (fast, well-studied)
""".strip()

In [ ]:
SCOUT_SYSTEM_PROMPT = """
You are Scout, an AI safety research agent specialised in mechanistic interpretability.

Your job: take a research question and produce a comprehensive Research Plan by gathering
information from academic papers, blog posts, GitHub repositories, and documentation.
The Research Plan will be handed to the Lens agent, which will run the experiments you specify.

## Workflow

Follow these steps in order:

1. **Decompose** — break the research question into 3-5 sub-topics
2. **Search arXiv** — run at least 3 targeted searches with different queries
3. **Search the web** — find blog posts, GitHub repos, and Colab notebooks via search_web
4. **Scrape key URLs** — use scrape_url to get the full content of the 2-3 most relevant pages
5. **Synthesise** — identify patterns, hypotheses, and open questions
6. **Save** — call `save_research_plan` with the final structured plan

## Research Plan Format

Your final output passed to `save_research_plan` MUST follow this exact structure:

```
# Research Plan: <research_question>

**Date**: <date>
**Status**: Draft

---

## Background
<2-3 paragraphs summarising the current state of knowledge on this topic>

## Key Papers & Resources
<bullet list — title, authors, year, one-line description, URL>

## Hypotheses
<numbered list of specific, testable hypotheses>

## Proposed Experiments
<For each experiment, use this exact format:>

### Experiment N: <name>
- **Lens Tool**: <one of: logit_lens | attention_pattern | ablation | activation_patching | direct_logit_attribution>
- **Model**: <e.g. gpt2-small, pythia-160m>
- **Dataset / Prompts**: <what inputs to use>
- **What to measure**: <specific metric or observation>
- **Hypothesis tested**: <which hypothesis above this tests>
- **Expected outcome**: <what you predict>

## Target Models
<which models and why — focus on TransformerLens-compatible small models>

## Expected Findings
<what the experiments are likely to reveal>

## Open Questions
<what remains unknown after this research plan is executed>
```

## Available Lens Tools (for Proposed Experiments)

| Tool | What it does |
|---|---|
| `logit_lens` | Visualises how predictions evolve across layers |
| `attention_pattern` | Shows which tokens attend to which at each layer |
| `ablation` | Zero/mean ablates heads or MLPs to measure their importance |
| `activation_patching` | Patches activations to find causal components |
| `direct_logit_attribution` | Decomposes final logit output by layer and component |

## Guidelines
- Focus on mechanistic interpretability and AI safety implications
- Prioritise papers from 2022 onwards (the modern mech interp era)
- Be concrete about experiments — Lens will run exactly what you specify
- Each hypothesis must be testable with TransformerLens on small models
- Prefer GPT-2 Small and Pythia models for experiments (fast, well-studied)
""".strip()

## 4. Build Scout Agent with LangGraph

We use `create_react_agent` — a prebuilt ReAct loop that:
1. Calls Claude with the available tools
2. If Claude calls a tool → executes it → feeds result back
3. Repeats until Claude produces a final answer

This mirrors Nova's `handle_agent_loop` but uses LangGraph's graph infrastructure instead of a manual `while True` loop.

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
# claude-opus-4-5 for best research planning quality
# swap to claude-sonnet-4-5 for faster / cheaper runs
model = ChatAnthropic(
    model="claude-opus-4-5",
    api_key=ANTHROPIC_API_KEY,
    temperature=1,
    max_tokens=8_000,
)

# ── Memory ────────────────────────────────────────────────────────────────────
# MemorySaver keeps the full message history in RAM across streamed chunks
memory = MemorySaver()

# ── Agent ─────────────────────────────────────────────────────────────────────
scout = create_react_agent(
    model=model,
    tools=SCOUT_TOOLS,
    checkpointer=memory,
    prompt=SystemMessage(content=SCOUT_SYSTEM_PROMPT),
)

print("✅ Scout agent ready")
print(f"   Model  : claude-opus-4-5")
print(f"   Tools  : {[t.name for t in SCOUT_TOOLS]}")
print(f"   Memory : MemorySaver (in-process)")

## 5. Run Scout

Edit `RESEARCH_QUESTION` below and run the cells. Scout will:
- Search arXiv and the web
- Scrape the most relevant URLs via Firecrawl
- Save a `research_plan.md` to the `outputs/` folder

In [ ]:
# ── Research Question ─────────────────────────────────────────────────────────
# Classic mech interp task — well-studied, good for validating the pipeline
RESEARCH_QUESTION = (
    "How does GPT-2 Small implement the Indirect Object Identification (IOI) task? "
    "Which attention heads and circuits are responsible, and what are the key "
    "computational mechanisms?"
)

# Thread ID — change this to start a fresh session
THREAD_ID = f"scout-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
config = {"configurable": {"thread_id": THREAD_ID}}

print(f"🔍 Research Question:")
print(f"   {RESEARCH_QUESTION}")
print(f"\n   Thread : {THREAD_ID}")
print("\n" + "=" * 70)
print("Running Scout...\n")

In [ ]:
# ── Stream the agent loop ─────────────────────────────────────────────────────
# stream_mode="updates" yields one dict per graph node update

for chunk in scout.stream(
    {"messages": [HumanMessage(content=RESEARCH_QUESTION)]},
    config=config,
    stream_mode="updates",
):
    for node_name, node_update in chunk.items():
        messages = node_update.get("messages", [])

        for msg in messages:
            # ── Tool call (agent deciding to use a tool) ───────────────────
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    args_preview = str(tc.get("args", {}))[:120]
                    print(f"🔧 [{tc['name']}] {args_preview}")

            # ── Tool result (result coming back from the tool) ─────────────
            elif hasattr(msg, "name") and msg.name in [t.name for t in SCOUT_TOOLS]:
                content_preview = str(msg.content)[:200].replace("\n", " ")
                print(f"   ↳ result: {content_preview}...")

            # ── Agent reasoning / final text ───────────────────────────────
            elif hasattr(msg, "content") and msg.content:
                content = msg.content
                if isinstance(content, list):
                    for block in content:
                        if isinstance(block, dict) and block.get("type") == "text":
                            preview = block["text"][:300].replace("\n", " ")
                            print(f"\n🤖 Scout: {preview}...\n")
                elif isinstance(content, str) and content.strip():
                    preview = content[:300].replace("\n", " ")
                    print(f"\n🤖 Scout: {preview}...\n")

print("\n" + "=" * 70)
print("✅ Scout run complete")

## 6. Display Research Plan

In [ ]:
from IPython.display import Markdown, display

plan_path = OUTPUTS_DIR / "research_plan.md"

if plan_path.exists():
    plan_text = plan_path.read_text(encoding="utf-8")
    print(f"📄 Research plan: {plan_path.resolve()}")
    print(f"   Size: {len(plan_text):,} chars\n")
    display(Markdown(plan_text))
else:
    print("⚠️  No research_plan.md found in outputs/.")
    print("    Scout may not have called save_research_plan yet.")
    print("    Check the streaming output above for errors.")

## 7. Inspect Agent State (Debug)

LangGraph's checkpointer stores the full message history. Useful for debugging what the agent saw.

In [ ]:
state    = scout.get_state(config)
messages = state.values.get("messages", [])

print(f"Total messages in thread: {len(messages)}\n")

for i, msg in enumerate(messages):
    role      = msg.__class__.__name__.replace("Message", "")
    tool_name = getattr(msg, "name", "")
    label     = f"[{role}]" if not tool_name else f"[Tool: {tool_name}]"

    content = msg.content
    if isinstance(content, list):
        preview = " | ".join(
            b.get("text", "")[:80] if isinstance(b, dict) else str(b)[:80]
            for b in content
        )
    else:
        preview = str(content)[:120].replace("\n", " ")

    print(f"{i:02d}  {label:<30} {preview}")

## 8. Next Steps

This notebook is a **draft** of Scout. Before wiring into the full Seesaw pipeline:

**Short term**
- [ ] Move tools into `scout/mcp_server/src/tools/` and `app/` following Nova's pattern
- [ ] Add `config/settings.py` — model selection, API key management (mirror Nova's `Settings` class)
- [ ] Add `config/constants.py` — file/folder name constants
- [ ] Add `config/prompts.py` — move system prompt here as a constant
- [ ] Add HITL checkpoint: pause after plan is saved, ask human to approve before Lens runs

**Tools to add**
- [ ] `search_semantic_scholar` — Semantic Scholar API (free, great AI coverage)
- [ ] `search_lesswrong` — LessWrong / Alignment Forum posts
- [ ] `process_github_urls` — gitingest for repos (port directly from Nova)
- [ ] `transcribe_youtube` — for recorded talks (port directly from Nova)

**Orchestrator integration**
- [ ] Expose as an MCP server (`scout/mcp_server/src/server.py`)
- [ ] Connect via `orchestrator/src/clients/scout_client.py`
- [ ] Wire HITL gate between Scout output and Lens input